# 확률적 경사 하강법

머신러닝 모델은 데이터로부터 학습하여 최적의 예측을 수행하는 것을 목표로 한다.\
이 과정에서 모델의 **가중치(weights)** 와 **편향(bias)** 을 조정하여 예측 오차를 최소화해야 하는데,\
이러한 최적화 과정의 핵심에 **경사 하강법(Gradient Descent)** 이 있다.\
경사 하강법은 손실 함수(loss function)의 기울기(gradient)를 계산하여 손실이 감소하는 방향으로 파라미터를 조금씩 업데이트하는 최적화 알고리즘이다.\
산을 내려갈 때 현재 위치에서 가장 가파른 내리막 방향으로 한 걸음씩 내려가는 것과 유사한 원리로 작동한다.\
그러나 전통적인 경사 하강법은 모든 훈련 데이터를 사용하여 한 번의 가중치 업데이트를 수행하기 때문에, \
데이터셋이 클 경우 다음과 같은 문제가 발생한다:

- 느린 학습 속도: 매 업데이트마다 전체 데이터를 처리해야 하므로 계산 비용이 크다
- 메모리 부족: 대용량 데이터를 한 번에 메모리에 로드하기 어렵다
- 지역 최솟값(local minimum) 탈출 어려움: 항상 같은 경로로 학습하여 최적해를 찾기 어려울 수 있다

이러한 한계를 극복하기 위해 등장한 것이 **확률적 경사 하강법(Stochastic Gradient Descent, SGD)** 이다. \
SGD는 전체 데이터 대신 무작위로 선택된 일부 샘플만을 사용하여 가중치를 업데이트한다. 이를 통해 다음과 같은 장점을 얻을 수 있다:

- 빠른 학습: 적은 데이터로 자주 업데이트하여 빠르게 수렴
- 메모리 효율성: 전체 데이터를 한 번에 로드할 필요 없음
- 지역 최솟값 탈출: 확률적 변동(stochastic noise)이 더 나은 해를 찾는 데 도움
본 문서에서는 확률적 경사 하강법의 작동 원리, 기존 경사 하강법과의 차이점, 그리고 실제 구현 방법에 대해 알아본다.
특히 **에포크(epoch)** 와 **미니배치(mini-batch)** 의 개념을 중심으로, 어떻게 효율적으로 모델을 학습시킬 수 있는지 살펴볼 것이다.

확률적 경사 하강법의 "확률적" 이라는 단어는 '무작위하게', 혹은 '랜덤하게'를 뜻한다.\
다음 "경사"는 언덕이나 산의 기울기를 나타낼 때 쓰는 '경사'와 같은 의미이다.(즉, 기울기를 나타낸다.)\
"하강법"은 내려가는 방법이다.

다시 말해, **확률적 경사 하강법(Stochastic Gradient Descent)** 는 **경사를 따라 내려가는 방법** 을 뜻한다.

산 꼭대기에서 내려오는 경우를 예로 들어, 경사 하강법은 가장 가파른 경사를 따라 가장 빠른 길을 선택한다.\
훈련 세트를 이용해 모델을 훈련하기 때문에 경사 하강법도 훈련 세트를 이용하여 가장 가파른 길을 찾는다.\
확률적 경사 하강법은 경사 하강법과 같은 원리이지만, 훈련 세트를 선택할 때 랜덤한 샘플을 선택한다.

확률적 경사 하강법은 훈련 세트에서 랜덤하게 하나의 샘플을 선택하여 가파른 경사를 조금 내려간다. 다음 훈련 세트에서 랜덤하게 또 다른 샘플을 하나 선택하여 경사를 조금 내려간다. 이러한 방식으로 전체 샘플을 모두 사용할 때까지 계속한다.\
이 일련의 과정을 **에포크(epoch)** 라고 부른다.\
경사 하강법에서 훈련 세트를 한 사이클 모두 사용하는 과정이다.\
일반적으로 경사 하강법은 수십, 수백 번 이상 에포크를 수행한다.
> 검증 데이터가 더 나아지지 않거나 overfitting이 시작되는 경우 에포크를 중단한다. loss 또한 에포크에 관여한다.


## 추가 - 신경망 알고리즘에서의 확률적 경사 하강법
확률적 경사 하강법은 신경망 알고리즘에서도 이용한다.\
신경망은 일반적으로 방대한 데이터를 사용하기 때문에 한 번에 모든 데이터를 한꺼번에 사용하기 어렵다. 또한 모델이 매우 복잡하기 때문에 수학적인 방법으로 해석하기 어렵다.

## 손실 함수
**손실 함수(loss function)** 는 특정 문제에서 머신러닝 알고리즘의 예측값과 실제 값의 차이를 수치화하는 함수이다.\
모델이 **얼마나 틀렸는지**를 측정하여 학습 과정에서 이를 최소화하는 것이 목표이다.

확률적 경사 하강법은 각 에포크를 수행하면서 각 에포크의 손실값을 확인하고, 이 손실값이 최소인 모델이 최적의 모델이라 볼 수 있다.\

분류에서 손실은 확실하게 정의할 수 있다.\
분류에서 정의한 정답을 그대로 맞추지 못하는 것으로 정의할 수 있다.\
4개의 이진 분류 샘플을 예시로 들면,
```
예측    정답
1   =   1
1   !=  0
0   =   0
0   !=  1
```
4개의 예측 중 2개만 맞았으므로 정확도는 $\frac{1}{2} = 0.5$이다.\
정확도를 손실 함수로 사용하는 것도 적절해 보이지만, 가능한 정확도와 개수는 정해져 있다.\
예시로 든 데이터 세트에서 가능한 정확도는 $0, 0.25, 0.5, 0.75, 1$의 5가지가 전부이다.

경사 하강법은 손실 함수의 기울기(gradient)를 계산하여 파라미터를 업데이트한다.\
다만, **정확도는 미분이 불가능**하므로 손실 함수로는 사용이 불가능하다.\
또한, 값의 차이가 각 $0.25$로 크므로 값을 조금만 변경해서는 정확도가 변하지 않는다.

손실 함수를 연속적으로 적절히 만드려면 다른 손실 함수를 사용해야 한다.

## 로지스틱 손실 함수(로그 손실(Log Loss))
`./04-1.ipynb`에서 사용한 로지스틱 회귀 모델의 확률을 이용할 때, 0~1 사이의 어떤 값도 가능하다.\
즉, **연속적이다.**\
위에서 사용한 4개의 예시 샘플의 예측 확률이 각각 $0.9, 0.2, 0.3, 0.8$이라고 가정해보자.
예측 값이 1에 가까울수록 양성 클래스이다.\

첫 번째 샘플의 예측은 $0.9$ 이므로 양성 클래스의 타깃인 1을 곱한 다음 음수로 바꾸면 $-0.9$이다.
두 번째 샘플의 예측은 $0.3$ 이다. 타깃인 양성 클래스(1)과는 거리가 멀다.\
첫 번째 샘플과 마찬가지로 타깃인 1을 곱한 다음 음수로 취하면 $-0.3$이다.

두 번째 샘플은 **타깃을 정확히 예측하지 못한, 즉, 손실이 큰 샘플**이다.

세 번째 샘플의 타깃은 음성 클래스이므로 0이다. 다만, 타깃값이 0이므로 어떤 수를 곱하던지 0이 된다.\
이를 해결하기 위해 타깃을 양성 클래스처럼 포장하여 1로 만드는 것이다. 예측값도 양성 클래스에 대한 예측으로 바뀐다.\
따라서, 양성 클래스에 대한 확률을 구하기 위해 1에서 샘플의 음성 클래스의 확률인 $0.3$을 뺀다.\
세 번째 샘플의 양성 클래스(타깃)에 대한 확률은 $0.7$이다. 곱하고 음수로 바꾸어 손실값을 계산하면 $-0.7$이다.

```
타깃    손실값
1(o)       -0.9
2(x)       -0.3
3(o)      -0.7
```
결과를 보면, 예측을 제대로 맞추지 못한 2번째 샘플의 손실이 가장 크다.\
똑같이 4번째 샘플의 확률을 예측하면, 음성 클래스이므로 타깃을 양성 클래스로 바꾸어 1에서 빼고 1을 곱해 음수로 취하면 손실값은 $-0.2$이다.
```
타깃    손실값
1(o)       -0.9
2(x)       -0.3 <= 손실이 크다.
3(o)      -0.7
4(x)      -0.2 <= 손실이 크다.
```
결과를 제대로 맞추지 못한 샘플에서의 손실이 큰 것을 확인할 수 있다.\
예측 확률을 이용해 이런 방식으로 계산하여 손실 함수를 얻을 수 있을 것이다.
## 표기법

## 1. 기본 형태

### 단일 샘플의 손실

$$
L(y, p) = -[y \log(p) + (1-y) \log(1-p)]
$$

여기서:
- $y$: 실제 레이블 (0 또는 1)
- $p$: 양성 클래스(1)일 예측 확률
- $1-p$: 음성 클래스(0)일 예측 확률

### 전체 데이터셋의 손실 (평균)

$$
\text{Log Loss} = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(p_i) + (1-y_i) \log(1-p_i)]
$$

여기서:
- $n$: 샘플 개수
- $y_i$: $i$번째 샘플의 실제 레이블
- $p_i$: $i$번째 샘플의 예측 확률

## 2. 다른 표기법들

### 표기법 1: 시그모이드 함수 명시

$$
\text{Log Loss} = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\sigma(z_i)) + (1-y_i) \log(1-\sigma(z_i))]
$$

여기서:
- $z_i = w^T x_i + b$ (로짓)
- $\sigma(z) = \frac{1}{1 + e^{-z}}$ (시그모이드 함수)

### 표기법 2: $\hat{y}$ 사용

$$
\text{Log Loss} = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i)]
$$

여기서:
- $\hat{y}_i$: 예측 확률 (= $p_i$)

### 표기법 3: 이진 크로스 엔트로피 (Binary Cross-Entropy)

$$
\text{BCE} = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(p_i) + (1-y_i) \log(1-p_i)]
$$

## 3. 경우별로 나누어 보기

### 실제 레이블이 1일 때 ($y=1$)

$$
L(1, p) = -[1 \times \log(p) + 0 \times \log(1-p)] = -\log(p)
$$

### 실제 레이블이 0일 때 ($y=0$)

$$
L(0, p) = -[0 \times \log(p) + 1 \times \log(1-p)] = -\log(1-p)
$$